## RAG Pipelines- Data Ingestions to Vector DB Pipeline ##

### Read All Pdf's inside Data FOlder ###

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os

In [32]:
def process_all_pdfs(directory):
    all_documents = []
    pdf_dir = Path(directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files Process..")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} Pages")
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal Documents: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files Process..

Processing: whitepaper_emebddings_vectorstores_v2.pdf
Loaded 64 Pages

Processing: attention.pdf
Loaded 15 Pages

Processing: Tech-Trends-in-Pratice-Esampler-1.pdf
Loaded 24 Pages

Total Documents: 103


In [34]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.2 (Macintosh)', 'creationdate': '2025-03-17T13:41:32-06:00', 'moddate': '2025-03-17T13:41:40-06:00', 'trapped': '/False', 'source': '../data/whitepaper_emebddings_vectorstores_v2.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'source_file': 'whitepaper_emebddings_vectorstores_v2.pdf', 'file_type': 'pdf'}, page_content='Embeddings  \n& Vector Stores\nAuthors: Anant Nawalgaria, \nXiaoqi Ren, and Charles Sugnet'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.2 (Macintosh)', 'creationdate': '2025-03-17T13:41:32-06:00', 'moddate': '2025-03-17T13:41:40-06:00', 'trapped': '/False', 'source': '../data/whitepaper_emebddings_vectorstores_v2.pdf', 'total_pages': 64, 'page': 1, 'page_label': '2', 'source_file': 'whitepaper_emebddings_vectorstores_v2.pdf', 'file_type': 'pdf'}, page_content='Embeddings & Vector Stores\n2\nFebrurary 2025\nContent contributors\nAnt

In [35]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
        )

    splits_docs = text_spliter.split_documents(documents)
    print(f"splits {len(documents)} into {len(splits_docs)} chunks")

    if splits_docs:
        print(f"\nExample Chunk:")
        print(f"content: {splits_docs[0].page_content[:200]}")
        print(f"metadata: {splits_docs[0].metadata}")
    return splits_docs

In [36]:
chunks = split_documents(all_pdf_documents)

splits 103 into 239 chunks

Example Chunk:
content: Embeddings  
& Vector Stores
Authors: Anant Nawalgaria, 
Xiaoqi Ren, and Charles Sugnet
metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.2 (Macintosh)', 'creationdate': '2025-03-17T13:41:32-06:00', 'moddate': '2025-03-17T13:41:40-06:00', 'trapped': '/False', 'source': '../data/whitepaper_emebddings_vectorstores_v2.pdf', 'total_pages': 64, 'page': 0, 'page_label': '1', 'source_file': 'whitepaper_emebddings_vectorstores_v2.pdf', 'file_type': 'pdf'}


### Embedding and VectorStoreDB ###